# BODAQS Macro Analysis Dashboard

This notebook drives the extracted backend (`bodaqs_analysis`) with minimal editing.

Workflow:
1. Load CSV
2. Normalize (zero + scale)
3. Estimate velocity/acceleration
4. Load event schema + detect events
5. Inspect metrics / quick plots


In [1]:
import pandas as pd
import numpy as np
import logging
from pathlib import Path

from bodaqs_analysis import run_macro
from bodaqs_analysis import normalize_and_scale
from bodaqs_analysis import estimate_va
from bodaqs_analysis import load_event_schema
from bodaqs_analysis import detect_events_from_schema
from bodaqs_analysis import extract_metrics_df

import importlib
import bodaqs_analysis.widgets.event_browser as eb
importlib.reload(eb)

from bodaqs_analysis.widgets.metric_histogram_widget import make_metric_histogram_widget
from bodaqs_analysis.widgets.event_browser import make_event_browser_widget_for_loader


logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)

# logging.getLogger("bodaqs_analysis").setLevel(logging.DEBUG)
logging.getLogger("bodaqs_analysis.detect").setLevel(logging.INFO)

logger = logging.getLogger(__name__)


In [2]:
# ---- Inputs ----
CSV_PATH = r"C:\Users\Ben Connor\OneDrive\Documents\logs\__2026-01-25*.CSV"   # change to your log
SCHEMA_PATH = r"event schema\event_schema.yaml"
INGESTION_MODE = "tolerant"
ZEROING_ENABLED = "false"

# Columns and full-ranges (mm) for normalization (edit as needed)
NORMALIZE_RANGES = {
    "front_shock_dom_suspension [mm]": 170.0,
    "rear_shock_dom_suspension [mm]": 165.0,
}

SAMPLE_RATE_HZ = 200   # if known; used for VA estimation when time col is absent


In [3]:
# ---- Batch orchestration ----

pattern = Path(CSV_PATH)
CSV_FILES = sorted(pattern.parent.glob(pattern.name))

if not CSV_FILES:
    raise FileNotFoundError(f"No CSV files matched: {CSV_PATH}")

logger.info(f"Found {len(CSV_FILES)} files:")
for p in CSV_FILES:
    logger.info("  %s", p.name)

events_all = []
metrics_all = []
results_all = []  # optional if you want to inspect per-file outputs
sessions_by_id = {}
schema = None

for p in CSV_FILES:
    logger.info(f"Processing {p.name} ...")
    results = run_macro(
        str(p),
        SCHEMA_PATH,
        zeroing_enabled=ZEROING_ENABLED,
        normalize_ranges=NORMALIZE_RANGES,
        sample_rate_hz=SAMPLE_RATE_HZ,
        strict = (INGESTION_MODE=="strict"),
    )
    session = results["session"]
    sid = str(session["session_id"])
    sessions_by_id[sid] = session
    #results_all.append((p, results))  # optional

    # pull from the correct keys
    ev = results["events"]
    mt = results["metrics"]
    if schema is None:
        schema = results["schema"]
    events_df = results["events"]

    # only append if non-empty (optional, but handy)
    if isinstance(ev, pd.DataFrame) and not ev.empty:
        events_all.append(ev)
    if isinstance(mt, pd.DataFrame) and not mt.empty:
        metrics_all.append(mt)

events_df  = pd.concat(events_all, ignore_index=True) if events_all else pd.DataFrame()
metrics_df = pd.concat(metrics_all, ignore_index=True) if metrics_all else pd.DataFrame()

logging.debug("events_df: %s", events_df.shape)
logging.debug("metrics_df: %s", metrics_df.shape)






INFO:__main__:Found 1 files:
INFO:__main__:  __2026-01-25_13-21-24.CSV
INFO:__main__:Processing __2026-01-25_13-21-24.CSV ...
INFO:bodaqs_analysis.pipeline:Session load complete: C:\Users\Ben Connor\OneDrive\Documents\logs\__2026-01-25_13-21-24.CSV
INFO:bodaqs_analysis.pipeline:Session pre-process complete
INFO:bodaqs_analysis.pipeline:Schema load complete
INFO:bodaqs_analysis.detect:Built EVENTS_DF with 107 rows from 2 schema event(s) → 4 sensor-expanded event(s)
INFO:bodaqs_analysis.pipeline:Event detection complete
INFO:bodaqs_analysis.pipeline:events rows: 107
INFO:bodaqs_analysis.pipeline:Running segment extraction for detected schema events: ['deep_compression', 'rebounds']
INFO:bodaqs_analysis.pipeline:Segment extraction complete (schema_id=deep_compression)
INFO:bodaqs_analysis.pipeline:segments valid (schema_id=deep_compression): 2/2
INFO:bodaqs_analysis.pipeline:Metrics calculation complete (schema_id=deep_compression)
INFO:bodaqs_analysis.pipeline:Segment extraction complete

In [4]:
make_metric_histogram_widget(
    events_df,
    metrics_df,
    event_type_col="event_name",
    signal_col="signal_col",
    default_bins=10,
)


INFO:bodaqs_analysis.widgets.metric_histogram_widget:metric_histogram: joined viz_df shape: (107, 9)


{'viz_df':                 session_id                       event_id  \
 0    __2026-01-25_13-21-24  deep_compression:rear_shock:0   
 1    __2026-01-25_13-21-24  deep_compression:rear_shock:1   
 2    __2026-01-25_13-21-24          rebounds:rear_shock:0   
 3    __2026-01-25_13-21-24          rebounds:rear_shock:1   
 4    __2026-01-25_13-21-24         rebounds:front_shock:0   
 ..                     ...                            ...   
 102  __2026-01-25_13-21-24         rebounds:rear_shock:58   
 103  __2026-01-25_13-21-24         rebounds:rear_shock:59   
 104  __2026-01-25_13-21-24         rebounds:rear_shock:60   
 105  __2026-01-25_13-21-24         rebounds:rear_shock:61   
 106  __2026-01-25_13-21-24        rebounds:front_shock:42   
 
                                      event_name  \
 0    deep compression (0.8 <= disp_norm < 0.98)   
 1    deep compression (0.8 <= disp_norm < 0.98)   
 2                       all rebound events >0.4   
 3                       all rebound

In [5]:
from bodaqs_analysis.widgets.event_browser import make_event_browser_widget_for_loader

def session_loader(session_id: str) -> dict:
    return sessions_by_id[str(session_id)]

wb = make_event_browser_widget_for_loader(
    schema,
    events_df,
    session_loader=session_loader,
)


In [6]:
from bodaqs_analysis.widgets.metric_scatter_widget import make_metric_scatter_widget

handles = make_metric_scatter_widget(
    events_df=events_df,
    metrics_df=metrics_df,
    event_type_col="event_name",   # or "event_type"/"schema_id" depending on your table
    signal_col="signal_col",
)

viz_df = handles["viz_df"]  # handy for debugging


INFO:bodaqs_analysis.widgets.metric_scatter_widget:metric_scatter: joined viz_df shape: (107, 10)


In [7]:
from bodaqs_analysis.widgets.signal_histogram_widget import (
    make_signal_histogram_widget_for_loader
)

def session_loader(session_id: str) -> dict:
    return sessions_by_id[str(session_id)]

wh = make_signal_histogram_widget_for_loader(
    events_df,
    session_loader=session_loader,
    default_bins=50,
)
